# UIT DSC 2026 LegalIR - Step 6 Fine-tuned Rerankers

Fine-tune two whitelisted reranker candidates on the same Step 5 semi-hard negatives: `itdainb/PhoRanker` with tokenizer resize/length clamp, and `AITeamVN/Vietnamese_Reranker`. Each passing candidate writes its own dev metrics and public `submission.zip`.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'safetensors', 'sentencepiece'], check=True)


## Config And Input Paths

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import time
import zipfile
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np

# Keep Step 6 stable on Kaggle T4x2. Reranker training on one T4 is enough for this baseline.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
OUTPUT_DIR = Path('/kaggle/working/step6')
PUBLIC_FILE = Path('/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json')
MODEL_NAME = 'itdainb/PhoRanker'
ALLOWED_MODELS = {
    'BAAI/bge-m3',
    'bkai-foundation-models/vietnamese-bi-encoder',
    'itdainb/PhoRanker',
    'BAAI/bge-reranker-v2-m3',
    'AITeamVN/Vietnamese_Reranker',
    'Qwen/Qwen3-Reranker-0.6B',
}
MAX_SUBMISSION_DOCS = 5

# Explicit input contract. Path aliases below only handle Kaggle folder nesting
# for the same artifacts; they never switch to another step's candidate pool.
def require_existing(name: str, candidates: list[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    msg = '\n'.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Missing required Step 6 input: {name}. Checked:\n{msg}')

STEP4_ROOTS = [DATA_ROOT / 'step4', DATA_ROOT / 'step4' / 'step4']
STEP5_ROOTS = [DATA_ROOT / 'step5', DATA_ROOT / 'step5' / 'step5', DATA_ROOT / 'step6' / 'step5']
CHUNKS_FILE = require_existing('step4/chunks.jsonl', [root / 'chunks.jsonl' for root in STEP4_ROOTS])
TRAIN_FILE = require_existing('step4/train_split.json', [root / 'train_split.json' for root in STEP4_ROOTS])
DEV_FILE = require_existing('step4/dev_split.json', [root / 'dev_split.json' for root in STEP4_ROOTS])
TRAIN_CANDIDATES = require_existing('step5/rankings/train_rankings_step5_fused.jsonl', [root / 'rankings' / 'train_rankings_step5_fused.jsonl' for root in STEP5_ROOTS])
STEP5_DEV_RANKINGS = require_existing('step5/rankings/dev_rankings_step5_fused.jsonl', [root / 'rankings' / 'dev_rankings_step5_fused.jsonl' for root in STEP5_ROOTS])
STEP5_PUBLIC_RANKINGS = require_existing('step5/rankings/public_rankings_step5_fused.jsonl', [root / 'rankings' / 'public_rankings_step5_fused.jsonl' for root in STEP5_ROOTS])
STEP5_RUN_REPORT = require_existing('step5/reports/run_report.json', [root / 'reports' / 'run_report.json' for root in STEP5_ROOTS])
STEP5_MODEL_MANIFEST = require_existing('step5/reports/model_manifest.json', [root / 'reports' / 'model_manifest.json' for root in STEP5_ROOTS])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for key, value in {
    'DATA_ROOT': DATA_ROOT,
    'CHUNKS_FILE': CHUNKS_FILE,
    'TRAIN_FILE': TRAIN_FILE,
    'DEV_FILE': DEV_FILE,
    'STEP5_DEV_RANKINGS': STEP5_DEV_RANKINGS,
    'STEP5_PUBLIC_RANKINGS': STEP5_PUBLIC_RANKINGS,
    'TRAIN_CANDIDATES': TRAIN_CANDIDATES,
    'OUTPUT_DIR': OUTPUT_DIR,
}.items():
    print(f'{key}: {value}')


## Utilities

In [ ]:
def read_json(path: Path) -> Any:
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, sort_keys=True)
        f.write('\n')

def iter_jsonl(path: Path) -> Iterable[dict[str, Any]]:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def append_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n')

def strip_accents(text: str) -> str:
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', text).replace('?', 'd').replace('?', 'D')

TOKEN_RE = re.compile(r'\b\w+\b', flags=re.UNICODE)
STOPWORDS = {
    'a','an','anh','ay','bi','boi','cac','can','cho','co','con','cua','duoc','da','de','den','di','do','doi','duoi','gi','hay','hoac','khi','la','lai','lam','mot','nay','neu','nhu','nhung','o','phai','qua','quy','rieng','sau','se','thi','theo','thuoc','toi','trong','tu','va','ve','vi','voi'
}

def tokenize(text: str) -> list[str]:
    text = strip_accents(text.lower())
    return [tok for tok in TOKEN_RE.findall(text) if len(tok) > 1 and tok not in STOPWORDS]

def evaluate_rankings(rankings: dict[str, list[str]], payload: dict[str, Any]) -> dict[str, Any]:
    rows = []
    for qid, row in payload.items():
        gold = [str(x) for x in row.get('answer', [])] if isinstance(row, dict) else []
        pred = [str(x) for x in rankings.get(str(qid), [])]
        top5 = pred[:MAX_SUBMISSION_DOCS]
        gold_set = set(gold)
        denom = len(gold_set) if gold_set else 1
        top5_hits = sum(1 for doc_id in top5 if doc_id in gold_set)
        hit_positions = [idx + 1 for idx, doc_id in enumerate(pred) if doc_id in gold_set]
        rows.append({
            'query_id': str(qid),
            'num_gold': len(gold_set),
            'exist@90': 1.0 if any(doc_id in gold_set for doc_id in pred[:90]) else 0.0,
            'hit@1': 1.0 if pred[:1] and pred[0] in gold_set else 0.0,
            'hit@5': 1.0 if top5_hits else 0.0,
            'hit@20': 1.0 if any(doc_id in gold_set for doc_id in pred[:20]) else 0.0,
            'mrr': 1.0 / hit_positions[0] if hit_positions else 0.0,
            'precision@5': top5_hits / MAX_SUBMISSION_DOCS,
            'recall@1': sum(1 for doc_id in pred[:1] if doc_id in gold_set) / denom,
            'recall@5': top5_hits / denom,
            'recall@20': sum(1 for doc_id in pred[:20] if doc_id in gold_set) / denom,
            'recall@50': sum(1 for doc_id in pred[:50] if doc_id in gold_set) / denom,
            'recall@90': sum(1 for doc_id in pred[:90] if doc_id in gold_set) / denom,
            'recall@100': sum(1 for doc_id in pred[:100] if doc_id in gold_set) / denom,
        })
    keys = [k for k in rows[0] if k not in {'query_id', 'num_gold'}] if rows else []
    return {'macro': {key: float(np.mean([row[key] for row in rows])) for key in keys}, 'per_query': rows}

def make_submission(predictions: dict[str, list[str]]) -> dict[str, dict[str, list[str]]]:
    return {str(qid): {'answer': [str(doc_id) for doc_id in docs[:MAX_SUBMISSION_DOCS]]} for qid, docs in predictions.items()}

def validate_submission_payload(submission: dict[str, Any], public_payload: dict[str, Any], valid_doc_ids: set[str]) -> dict[str, Any]:
    issues = []
    expected_qids = set(map(str, public_payload.keys()))
    actual_qids = set(map(str, submission.keys()))
    for qid in sorted(expected_qids - actual_qids):
        issues.append({'level': 'error', 'query_id': qid, 'message': 'missing query_id'})
    for qid in sorted(actual_qids - expected_qids):
        issues.append({'level': 'error', 'query_id': qid, 'message': 'unexpected query_id'})
    lengths = Counter()
    for qid, row in submission.items():
        answer = row.get('answer') if isinstance(row, dict) else None
        if not isinstance(answer, list):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'answer must be a list'})
            continue
        lengths[str(len(answer))] += 1
        if not (1 <= len(answer) <= MAX_SUBMISSION_DOCS):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'answer length must be 1..5'})
        if len(answer) != len(set(answer)):
            issues.append({'level': 'error', 'query_id': str(qid), 'message': 'duplicate document_id'})
        for doc_id in answer:
            if not isinstance(doc_id, str):
                issues.append({'level': 'error', 'query_id': str(qid), 'message': 'document_id must be string'})
            elif doc_id not in valid_doc_ids:
                issues.append({'level': 'error', 'query_id': str(qid), 'message': f'unknown document_id: {doc_id}'})
    return {
        'num_public_queries': len(expected_qids),
        'num_submission_queries': len(actual_qids),
        'answer_length_distribution': dict(sorted(lengths.items())),
        'num_errors': sum(1 for issue in issues if issue['level'] == 'error'),
        'num_warnings': sum(1 for issue in issues if issue['level'] == 'warning'),
        'issues': issues[:200],
    }

def write_submission_zip(submission_json: Path, zip_path: Path) -> None:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(submission_json, arcname='submission.json')


## Load Corpus And Candidate Rankings

In [ ]:
def load_chunks(chunks_file: Path) -> tuple[list[dict[str, Any]], dict[str, list[int]], set[str]]:
    chunks = []
    doc_to_chunk_indices: dict[str, list[int]] = defaultdict(list)
    valid_doc_ids = set()
    for idx, row in enumerate(iter_jsonl(chunks_file)):
        doc_id = str(row.get('doc_id', ''))
        chunk = {
            'chunk_idx': idx,
            'chunk_id': str(row.get('chunk_id', idx)),
            'doc_id': doc_id,
            'text': str(row.get('text', '')),
            'heading': str(row.get('heading', '')),
            'word_count': int(row.get('word_count') or 0),
            'metadata': row.get('metadata') if isinstance(row.get('metadata'), dict) else {},
        }
        chunks.append(chunk)
        if doc_id:
            doc_to_chunk_indices[doc_id].append(idx)
            valid_doc_ids.add(doc_id)
        if (idx + 1) % 50000 == 0:
            print(f'loaded {idx + 1:,} chunks')
    return chunks, doc_to_chunk_indices, valid_doc_ids

def load_rankings(path: Path) -> dict[str, list[str]]:
    rankings = {}
    for row in iter_jsonl(path):
        qid = str(row.get('query_id'))
        if 'fused_doc_ids' in row:
            rankings[qid] = [str(x) for x in row['fused_doc_ids']]
        elif 'top_docs' in row:
            rankings[qid] = [str(x.get('doc_id')) for x in row['top_docs']]
        elif 'doc_ids' in row:
            rankings[qid] = [str(x) for x in row['doc_ids']]
    return rankings

def pick_evidence_text(question: str, doc_id: str, *, max_chunks: int = 3, max_chars: int = 1800) -> tuple[str, list[str]]:
    q = Counter(tokenize(question))
    q_set = set(q)
    scored = []
    for chunk_idx in doc_to_chunk_indices.get(str(doc_id), []):
        chunk = chunks[chunk_idx]
        toks = tokenize((chunk['heading'] + ' ' + chunk['text'])[:7000])
        cc = Counter(toks)
        overlap = sum(min(q[tok], cc[tok]) for tok in q_set)
        heading_bonus = 0.25 if any(tok in strip_accents(chunk['heading'].lower()) for tok in q_set) else 0.0
        score = overlap / max(1, len(q_set)) + heading_bonus
        scored.append((score, -abs(chunk['word_count'] - 320), chunk_idx))
    scored.sort(reverse=True)
    parts = []
    chunk_ids = []
    for _, _, chunk_idx in scored[:max_chunks]:
        chunk = chunks[chunk_idx]
        chunk_ids.append(chunk['chunk_id'])
        parts.append((chunk['heading'] + '\n' + chunk['text']).strip())
    text = '\n\n'.join(parts)
    if len(text) > max_chars:
        text = text[:max_chars]
    return text, chunk_ids

train_payload = read_json(TRAIN_FILE)
dev_payload = read_json(DEV_FILE)
public_payload = read_json(PUBLIC_FILE)
chunks, doc_to_chunk_indices, valid_doc_ids = load_chunks(CHUNKS_FILE)
train_candidates = load_rankings(TRAIN_CANDIDATES)
dev_rankings_base = load_rankings(STEP5_DEV_RANKINGS)
public_rankings_base = load_rankings(STEP5_PUBLIC_RANKINGS)
print(f'chunks={len(chunks):,}; docs={len(valid_doc_ids):,}; train_candidates={len(train_candidates):,}; dev={len(dev_rankings_base):,}; public={len(public_rankings_base):,}')


## Build Semi-hard Reranker Samples

In [ ]:
@dataclass(frozen=True)
class Step6Config:
    max_seq_length: int = 384
    train_top_candidates: int = 80
    rerank_top_docs: int = 50
    negatives_per_positive: int = 10
    max_train_queries: int = 6000
    max_train_pairs: int = 80000
    train_batch_size: int = 8
    grad_accum_steps: int = 2
    epochs: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    evidence_chunks: int = 3
    evidence_max_chars: int = 1800
    rerank_batch_size: int = 32
    retrieval_weights: tuple[float, ...] = (0.2, 0.3, 0.4, 0.5)
    reranker_weights: tuple[float, ...] = (0.5, 0.6, 0.7, 0.8)
    finetune_model_ids: tuple[str, ...] = ('itdainb/PhoRanker', 'AITeamVN/Vietnamese_Reranker')

config = Step6Config()
write_json(OUTPUT_DIR / 'configs' / 'step6_config.json', asdict(config))

def build_training_rows() -> list[dict[str, Any]]:
    cache_file = OUTPUT_DIR / 'training' / 'phoranker_train_pairs.jsonl'
    if cache_file.exists():
        rows = list(iter_jsonl(cache_file))
        if len(rows) >= 1000:
            print(json.dumps({'loaded_cached_train_pairs': len(rows), 'cache_file': str(cache_file)}, ensure_ascii=False, indent=2))
            return rows
        print(json.dumps({'ignored_small_train_pair_cache': len(rows), 'cache_file': str(cache_file)}, ensure_ascii=False, indent=2))
    rows = []
    stats = Counter()
    q_items = list(train_payload.items())[:config.max_train_queries]
    rng = random.Random(SEED)
    for qid, row in q_items:
        qid = str(qid)
        question = row.get('question', '') if isinstance(row, dict) else ''
        gold_docs = [str(x) for x in row.get('answer', [])] if isinstance(row, dict) else []
        gold_set = set(gold_docs)
        candidates = [doc_id for doc_id in train_candidates.get(qid, []) if doc_id]
        neg_pool = []
        seen = set()
        for doc_id in candidates[:config.train_top_candidates]:
            if doc_id in gold_set or doc_id in seen:
                continue
            seen.add(doc_id)
            neg_pool.append(doc_id)
        if not neg_pool:
            stats['no_neg_pool'] += 1
            continue
        # semi-hard: skip the very first candidates; do not fall back to hard-top negatives.
        semi_hard = neg_pool[5: min(len(neg_pool), 60)]
        if not semi_hard:
            stats['no_semi_hard_pool'] += 1
            continue
        for gold_doc in gold_docs:
            if gold_doc not in valid_doc_ids:
                stats['missing_gold_doc'] += 1
                continue
            pos_text, pos_chunks = pick_evidence_text(question, gold_doc, max_chunks=config.evidence_chunks, max_chars=config.evidence_max_chars)
            if not pos_text:
                stats['empty_positive'] += 1
                continue
            rows.append({'query_id': qid, 'question': question, 'doc_id': gold_doc, 'text': pos_text, 'label': 1.0, 'chunk_ids': pos_chunks})
            chosen_negs = rng.sample(semi_hard, k=min(config.negatives_per_positive, len(semi_hard)))
            for neg_doc in chosen_negs:
                neg_text, neg_chunks = pick_evidence_text(question, neg_doc, max_chunks=config.evidence_chunks, max_chars=config.evidence_max_chars)
                if not neg_text:
                    stats['empty_negative'] += 1
                    continue
                rows.append({'query_id': qid, 'question': question, 'doc_id': neg_doc, 'text': neg_text, 'label': 0.0, 'chunk_ids': neg_chunks})
    rng.shuffle(rows)
    if config.max_train_pairs and len(rows) > config.max_train_pairs:
        rows = rows[:config.max_train_pairs]
    stats['num_rows'] = len(rows)
    stats['num_positive'] = sum(1 for r in rows if r['label'] == 1.0)
    stats['num_negative'] = sum(1 for r in rows if r['label'] == 0.0)
    write_json(OUTPUT_DIR / 'training' / 'training_pair_report.json', dict(stats))
    append_jsonl(OUTPUT_DIR / 'training' / 'phoranker_train_pairs.jsonl', rows)
    print(json.dumps(dict(stats), ensure_ascii=False, indent=2))
    return rows

train_rows = build_training_rows()


## Fine-tune Selected Rerankers


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

RERANKER_CANDIDATES = [
    {
        'model_id': 'AITeamVN/Vietnamese_Reranker',
        'slug': 'aiteamvn_vietnamese_reranker_finetuned',
        'finetune': True,
        'allow_resize': False,
        'use_fast': False,
        'trust_remote_code': True,
    },
    {
        'model_id': 'itdainb/PhoRanker',
        'slug': 'phoranker_resized_finetuned',
        'finetune': True,
        'allow_resize': True,
        'use_fast': False,
        'trust_remote_code': True,
    },
]

class PairDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        encoded = self.tokenizer(
            row['question'],
            row['text'],
            truncation='only_second',
            padding='max_length',
            max_length=self.max_length,
            return_tensors=None,
        )
        item = {key: torch.tensor(value, dtype=torch.long) for key, value in encoded.items()}
        item['labels'] = torch.tensor(float(row['label']), dtype=torch.float32)
        return item

def logits_to_relevance(logits: torch.Tensor) -> torch.Tensor:
    if logits.ndim == 1 or logits.shape[-1] == 1:
        return logits.view(-1)
    if logits.shape[-1] == 2:
        return logits[:, 1] - logits[:, 0]
    return logits.max(dim=-1).values

def validate_batch_token_ids(batch: dict[str, torch.Tensor], *, model_vocab_size: int, tokenizer_len: int, where: str) -> None:
    input_ids = batch.get('input_ids')
    if input_ids is None:
        return
    min_id = int(input_ids.min().item())
    max_id = int(input_ids.max().item())
    if min_id < 0 or max_id >= model_vocab_size:
        raise ValueError(f'Invalid token id at {where}: min={min_id}, max={max_id}, model_vocab_size={model_vocab_size}, tokenizer_len={tokenizer_len}')

def load_candidate(candidate: dict[str, Any], *, device: torch.device) -> dict[str, Any]:
    model_id = candidate['model_id']
    if model_id not in ALLOWED_MODELS:
        raise ValueError(f'Model not whitelisted: {model_id}')
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=candidate.get('use_fast', False), trust_remote_code=candidate.get('trust_remote_code', True))
    model = AutoModelForSequenceClassification.from_pretrained(model_id, trust_remote_code=candidate.get('trust_remote_code', True))
    embedding_size = model.get_input_embeddings().num_embeddings
    resized_from_to = None
    if len(tokenizer) > embedding_size:
        if not candidate.get('allow_resize', False):
            raise ValueError(f'Tokenizer/model vocab mismatch for {model_id}: tokenizer_len={len(tokenizer)}, embedding_size={embedding_size}, resize disabled')
        resized_from_to = {'from': embedding_size, 'to': len(tokenizer)}
        print({'candidate': candidate['slug'], 'resize_token_embeddings': resized_from_to})
        model.resize_token_embeddings(len(tokenizer))
    model_vocab_size = model.get_input_embeddings().num_embeddings
    max_positions = getattr(model.config, 'max_position_embeddings', None)
    if isinstance(max_positions, int) and max_positions > 2:
        effective_max_length = min(config.max_seq_length, max_positions - 2)
    else:
        effective_max_length = config.max_seq_length
    if effective_max_length < config.max_seq_length:
        print({'candidate': candidate['slug'], 'clamp_max_seq_length': {'from': config.max_seq_length, 'to': effective_max_length, 'model_max_position_embeddings': max_positions}})
    sanity_rows = train_rows[:2] if train_rows else [{'question': 'test', 'text': 'test'}]
    encoded = tokenizer(
        [r['question'] for r in sanity_rows],
        [r['text'] for r in sanity_rows],
        truncation='only_second',
        padding=True,
        max_length=effective_max_length,
        return_tensors='pt',
    )
    validate_batch_token_ids(encoded, model_vocab_size=model_vocab_size, tokenizer_len=len(tokenizer), where=f"{candidate['slug']} cpu sanity")
    model.eval()
    with torch.inference_mode():
        _ = logits_to_relevance(model(**encoded).logits)
    model.to(device)
    param_count = int(sum(p.numel() for p in model.parameters()))
    info = {
        'candidate': candidate['slug'],
        'model_id': model_id,
        'finetune': bool(candidate.get('finetune')),
        'tokenizer_len': len(tokenizer),
        'model_vocab_size': model_vocab_size,
        'resized_token_embeddings': resized_from_to,
        'max_position_embeddings': max_positions,
        'effective_max_seq_length': effective_max_length,
        'parameter_count': param_count,
        'device': str(device),
    }
    print(json.dumps(info, ensure_ascii=False, indent=2))
    return {'candidate': candidate, 'tokenizer': tokenizer, 'model': model, 'model_vocab_size': model_vocab_size, 'max_length': effective_max_length, 'info': info}

def train_candidate(bundle: dict[str, Any]) -> tuple[Path | None, list[dict[str, float]]]:
    candidate = bundle['candidate']
    if not candidate.get('finetune', False):
        return None, []
    model = bundle['model']
    tokenizer = bundle['tokenizer']
    model_vocab_size = bundle['model_vocab_size']
    slug = candidate['slug']
    train_dataset = PairDataset(train_rows, tokenizer, bundle['max_length'])
    for probe_idx in range(min(32, len(train_dataset))):
        probe = train_dataset[probe_idx]
        validate_batch_token_ids({'input_ids': probe['input_ids']}, model_vocab_size=model_vocab_size, tokenizer_len=len(tokenizer), where=f'{slug} preflight row {probe_idx}')
    train_loader = DataLoader(train_dataset, batch_size=config.train_batch_size, shuffle=True, num_workers=0, pin_memory=(device.type == 'cuda'))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = math.ceil(len(train_loader) / config.grad_accum_steps) * config.epochs
    warmup_steps = math.ceil(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
    criterion = torch.nn.BCEWithLogitsLoss()
    model.train()
    global_step = 0
    loss_history = []
    for epoch in range(config.epochs):
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        for step, batch in enumerate(train_loader, start=1):
            labels = batch.pop('labels').to(device)
            validate_batch_token_ids(batch, model_vocab_size=model_vocab_size, tokenizer_len=len(tokenizer), where=f'{slug} epoch {epoch + 1} step {step}')
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
                logits = logits_to_relevance(model(**batch).logits)
                loss = criterion(logits, labels) / config.grad_accum_steps
            scaler.scale(loss).backward()
            running += float(loss.detach().cpu()) * config.grad_accum_steps
            if step % config.grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
            if step % 250 == 0:
                print({'candidate': slug, 'epoch': epoch + 1, 'step': step, 'avg_loss': round(running / step, 5)})
        epoch_loss = running / max(1, len(train_loader))
        loss_history.append({'epoch': epoch + 1, 'loss': epoch_loss})
        print({'candidate': slug, 'epoch': epoch + 1, 'loss': epoch_loss})
    model_output_path = OUTPUT_DIR / 'models' / slug
    model_output_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(model_output_path)
    tokenizer.save_pretrained(model_output_path)
    write_json(OUTPUT_DIR / 'metrics' / slug / 'train_loss_history.json', loss_history)
    print('Saved model:', model_output_path)
    return model_output_path, loss_history

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
write_json(OUTPUT_DIR / 'configs' / 'reranker_candidates.json', RERANKER_CANDIDATES)
print({'device': str(device), 'candidates': [c['model_id'] for c in RERANKER_CANDIDATES]})


## Rerank Dev And Tune Fusion

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-float(x)))

@torch.inference_mode()
def score_pairs(bundle: dict[str, Any], pair_rows: list[dict[str, Any]]) -> list[float]:
    model = bundle['model']
    tokenizer = bundle['tokenizer']
    model_vocab_size = bundle['model_vocab_size']
    model.eval()
    scores = []
    for start in range(0, len(pair_rows), config.rerank_batch_size):
        batch_rows = pair_rows[start:start + config.rerank_batch_size]
        encoded = tokenizer(
            [r['question'] for r in batch_rows],
            [r['text'] for r in batch_rows],
            truncation='only_second',
            padding=True,
            max_length=bundle['max_length'],
            return_tensors='pt',
        )
        validate_batch_token_ids(encoded, model_vocab_size=model_vocab_size, tokenizer_len=len(tokenizer), where=f"{bundle['candidate']['slug']} score batch {start}")
        encoded = {k: v.to(device) for k, v in encoded.items()}
        logits = logits_to_relevance(model(**encoded).logits).detach().cpu().numpy().tolist()
        scores.extend([sigmoid(x) for x in logits])
    return scores

def minmax(values: list[float]) -> list[float]:
    if not values:
        return []
    lo, hi = min(values), max(values)
    if abs(hi - lo) < 1e-12:
        return [0.5 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]

def rerank_payload(bundle: dict[str, Any], payload: dict[str, Any], base_rankings: dict[str, list[str]], output_file: Path) -> tuple[dict[str, list[str]], dict[str, list[dict[str, Any]]]]:
    slug = bundle['candidate']['slug']
    all_rankings = {}
    scored_rows_by_qid = {}
    def out_rows():
        for idx, (qid, row) in enumerate(payload.items(), start=1):
            qid = str(qid)
            question = row.get('question', '') if isinstance(row, dict) else ''
            candidates = base_rankings.get(qid, [])[:config.rerank_top_docs]
            pair_rows = []
            for rank, doc_id in enumerate(candidates, start=1):
                evidence_text, chunk_ids = pick_evidence_text(question, doc_id, max_chunks=config.evidence_chunks, max_chars=config.evidence_max_chars)
                pair_rows.append({'query_id': qid, 'question': question, 'doc_id': doc_id, 'text': evidence_text, 'retrieval_rank': rank, 'chunk_ids': chunk_ids})
            reranker_scores = score_pairs(bundle, pair_rows) if pair_rows else []
            scored = []
            for pair, score in zip(pair_rows, reranker_scores):
                scored.append({
                    'doc_id': pair['doc_id'],
                    'reranker_score': score,
                    'retrieval_rank': pair['retrieval_rank'],
                    'chunk_ids': pair['chunk_ids'],
                })
            scored.sort(key=lambda r: r['reranker_score'], reverse=True)
            reranked = [r['doc_id'] for r in scored]
            tail = [doc_id for doc_id in base_rankings.get(qid, []) if doc_id not in set(reranked)]
            all_rankings[qid] = reranked + tail
            scored_rows_by_qid[qid] = scored
            if idx % 100 == 0:
                print(f'{slug}: reranked {idx:,}/{len(payload):,}')
            yield {'query_id': qid, 'question': question, 'gold': row.get('answer') if isinstance(row, dict) else None, 'reranked_top_docs': scored, 'reranked_doc_ids': all_rankings[qid]}
    append_jsonl(output_file, out_rows())
    return all_rankings, scored_rows_by_qid

def fuse_with_scores(base_rankings: dict[str, list[str]], reranked_scores: dict[str, list[dict[str, Any]]], *, retrieval_weight: float, reranker_weight: float) -> dict[str, list[str]]:
    fused = {}
    for qid, base_docs in base_rankings.items():
        top_docs = base_docs[:config.rerank_top_docs]
        retrieval_raw = [1.0 / rank for rank in range(1, len(top_docs) + 1)]
        retrieval_norm = dict(zip(top_docs, minmax(retrieval_raw)))
        reranker_raw = {row['doc_id']: row['reranker_score'] for row in reranked_scores.get(qid, [])}
        reranker_norm_vals = minmax(list(reranker_raw.values()))
        reranker_norm = dict(zip(reranker_raw.keys(), reranker_norm_vals))
        scores = {}
        first_seen = {}
        for idx, doc_id in enumerate(top_docs):
            first_seen.setdefault(doc_id, idx)
            scores[doc_id] = retrieval_weight * retrieval_norm.get(doc_id, 0.0) + reranker_weight * reranker_norm.get(doc_id, 0.0)
        for idx, doc_id in enumerate(base_docs[config.rerank_top_docs:], start=len(top_docs)):
            first_seen.setdefault(doc_id, idx)
            scores.setdefault(doc_id, -idx * 1e-6)
        fused[qid] = [doc_id for doc_id, _ in sorted(scores.items(), key=lambda item: (-item[1], first_seen[item[0]]))[:100]]
    return fused

def run_candidate(candidate: dict[str, Any]) -> dict[str, Any]:
    slug = candidate['slug']
    started = time.time()
    candidate_dir = OUTPUT_DIR / 'candidates' / slug
    metrics_dir = OUTPUT_DIR / 'metrics' / slug
    rankings_dir = OUTPUT_DIR / 'rankings' / slug
    submission_dir = OUTPUT_DIR / 'submission' / slug
    for path in [candidate_dir, metrics_dir, rankings_dir, submission_dir]:
        path.mkdir(parents=True, exist_ok=True)
    bundle = load_candidate(candidate, device=device)
    model_output_path, loss_history = train_candidate(bundle)

    dev_reranked, dev_reranker_scores = rerank_payload(bundle, dev_payload, dev_rankings_base, rankings_dir / 'dev_rankings_reranker.jsonl')
    reranker_only_metrics = evaluate_rankings(dev_reranked, dev_payload)
    write_json(metrics_dir / 'dev_metrics_reranker_only.json', reranker_only_metrics)

    trials = []
    for retrieval_weight in config.retrieval_weights:
        for reranker_weight in config.reranker_weights:
            fused = fuse_with_scores(dev_rankings_base, dev_reranker_scores, retrieval_weight=retrieval_weight, reranker_weight=reranker_weight)
            metrics = evaluate_rankings(fused, dev_payload)['macro']
            trials.append({'retrieval_weight': retrieval_weight, 'reranker_weight': reranker_weight, 'metrics': metrics})
    trials.sort(key=lambda r: (r['metrics']['recall@5'], r['metrics']['precision@5'], r['metrics']['recall@20'], r['metrics']['mrr']), reverse=True)
    best_fusion = {k: trials[0][k] for k in ['retrieval_weight', 'reranker_weight']}
    write_json(metrics_dir / 'fusion_trials.json', trials)
    write_json(candidate_dir / 'best_fusion_config.json', best_fusion)

    dev_fused = fuse_with_scores(dev_rankings_base, dev_reranker_scores, **best_fusion)
    dev_metrics = evaluate_rankings(dev_fused, dev_payload)
    write_json(metrics_dir / 'dev_metrics_step6_fused.json', dev_metrics)
    def final_dev_rows():
        for qid, row in dev_payload.items():
            qid = str(qid)
            yield {'query_id': qid, 'question': row.get('question', ''), 'gold': row.get('answer'), 'base_doc_ids': dev_rankings_base.get(qid, []), 'fused_doc_ids': dev_fused.get(qid, [])}
    append_jsonl(rankings_dir / 'dev_rankings_step6_fused.jsonl', final_dev_rows())
    write_json(OUTPUT_DIR / 'predictions' / f'dev_predictions_top5_{slug}.json', {qid: docs[:MAX_SUBMISSION_DOCS] for qid, docs in dev_fused.items()})

    public_reranked, public_reranker_scores = rerank_payload(bundle, public_payload, public_rankings_base, rankings_dir / 'public_rankings_reranker.jsonl')
    public_fused = fuse_with_scores(public_rankings_base, public_reranker_scores, **best_fusion)
    def final_public_rows():
        for qid, row in public_payload.items():
            qid = str(qid)
            yield {'query_id': qid, 'question': row.get('question', ''), 'gold': None, 'base_doc_ids': public_rankings_base.get(qid, []), 'fused_doc_ids': public_fused.get(qid, [])}
    append_jsonl(rankings_dir / 'public_rankings_step6_fused.jsonl', final_public_rows())

    submission = make_submission({qid: docs[:MAX_SUBMISSION_DOCS] for qid, docs in public_fused.items()})
    validation = validate_submission_payload(submission, public_payload, valid_doc_ids)
    write_json(submission_dir / 'submission.json', submission)
    write_json(submission_dir / 'submission_validation.json', validation)
    if validation['num_errors']:
        raise ValueError(f"Submission validation failed for {slug}: {validation['num_errors']} errors")
    write_submission_zip(submission_dir / 'submission.json', submission_dir / 'submission.zip')

    result = {
        'status': 'ok',
        'candidate': candidate,
        'model_info': bundle['info'],
        'model_output_path': str(model_output_path) if model_output_path else None,
        'loss_history': loss_history,
        'best_fusion': best_fusion,
        'reranker_only_dev_macro': reranker_only_metrics['macro'],
        'fused_dev_macro': dev_metrics['macro'],
        'seconds': round(time.time() - started, 3),
        'outputs': {
            'submission_zip': str(submission_dir / 'submission.zip'),
            'submission_validation': str(submission_dir / 'submission_validation.json'),
            'dev_rankings': str(rankings_dir / 'dev_rankings_step6_fused.jsonl'),
            'public_rankings': str(rankings_dir / 'public_rankings_step6_fused.jsonl'),
        },
    }
    write_json(candidate_dir / 'run_report.json', result)
    print(json.dumps({'candidate': slug, 'fused_dev_macro': dev_metrics['macro'], 'submission_zip': result['outputs']['submission_zip']}, ensure_ascii=False, indent=2))
    del bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

candidate_results = []
for candidate in RERANKER_CANDIDATES:
    try:
        candidate_results.append(run_candidate(candidate))
    except Exception as exc:
        failure = {
            'status': 'failed',
            'candidate': candidate,
            'error_type': type(exc).__name__,
            'error': str(exc),
        }
        candidate_results.append(failure)
        write_json(OUTPUT_DIR / 'candidates' / candidate['slug'] / 'run_report.json', failure)
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

ok_results = [r for r in candidate_results if r.get('status') == 'ok']
if not ok_results:
    raise RuntimeError('No reranker candidate completed successfully')
summary = sorted(
    ok_results,
    key=lambda r: (r['fused_dev_macro']['recall@5'], r['fused_dev_macro']['precision@5'], r['fused_dev_macro']['mrr']),
    reverse=True,
)
write_json(OUTPUT_DIR / 'reports' / 'candidate_summary.json', candidate_results)
write_json(OUTPUT_DIR / 'reports' / 'candidate_summary_sorted.json', summary)
print('Candidate summary sorted by dev Recall@5:')
print(json.dumps([
    {
        'slug': r['candidate']['slug'],
        'model_id': r['candidate']['model_id'],
        'finetune': r['candidate']['finetune'],
        'recall@5': r['fused_dev_macro']['recall@5'],
        'precision@5': r['fused_dev_macro']['precision@5'],
        'mrr': r['fused_dev_macro']['mrr'],
        'submission_zip': r['outputs']['submission_zip'],
    }
    for r in summary
], ensure_ascii=False, indent=2))


## Rerank Public And Create Submission

In [ ]:
best_result = summary[0]
report = {
    'inputs': {
        'chunks_file': str(CHUNKS_FILE),
        'train_file': str(TRAIN_FILE),
        'dev_file': str(DEV_FILE),
        'train_candidates': str(TRAIN_CANDIDATES),
        'step5_dev_rankings': str(STEP5_DEV_RANKINGS),
        'step5_public_rankings': str(STEP5_PUBLIC_RANKINGS),
        'step5_run_report': str(STEP5_RUN_REPORT),
        'step5_model_manifest': str(STEP5_MODEL_MANIFEST),
    },
    'config': asdict(config),
    'allowed_models': sorted(ALLOWED_MODELS),
    'candidate_results': candidate_results,
    'best_by_dev_recall5': {
        'slug': best_result['candidate']['slug'],
        'model_id': best_result['candidate']['model_id'],
        'finetune': best_result['candidate']['finetune'],
        'fused_dev_macro': best_result['fused_dev_macro'],
        'submission_zip': best_result['outputs']['submission_zip'],
    },
    'public_submission_zips': {
        r['candidate']['slug']: r['outputs']['submission_zip']
        for r in candidate_results
        if r.get('status') == 'ok'
    },
    'next_step7_inputs': [
        'step6/reports/candidate_summary_sorted.json',
        'step6/candidates/<slug>/best_fusion_config.json',
        'step6/rankings/<slug>/dev_rankings_step6_fused.jsonl',
        'step6/rankings/<slug>/public_rankings_step6_fused.jsonl',
    ],
}
write_json(OUTPUT_DIR / 'reports' / 'run_report.json', report)
print(json.dumps(report['best_by_dev_recall5'], ensure_ascii=False, indent=2))
print('Submission zips:')
for slug, path in report['public_submission_zips'].items():
    print(slug, path)


## Files To Download

In [ ]:
for path in [
    OUTPUT_DIR / 'reports' / 'run_report.json',
    OUTPUT_DIR / 'reports' / 'candidate_summary.json',
    OUTPUT_DIR / 'reports' / 'candidate_summary_sorted.json',
    OUTPUT_DIR / 'configs' / 'reranker_candidates.json',
]:
    print(path, 'OK' if path.exists() else 'MISSING')

print('\nCandidate outputs:')
for candidate in RERANKER_CANDIDATES:
    slug = candidate['slug']
    for path in [
        OUTPUT_DIR / 'candidates' / slug / 'run_report.json',
        OUTPUT_DIR / 'submission' / slug / 'submission.zip',
        OUTPUT_DIR / 'submission' / slug / 'submission_validation.json',
        OUTPUT_DIR / 'metrics' / slug / 'dev_metrics_step6_fused.json',
        OUTPUT_DIR / 'metrics' / slug / 'dev_metrics_reranker_only.json',
        OUTPUT_DIR / 'metrics' / slug / 'fusion_trials.json',
        OUTPUT_DIR / 'rankings' / slug / 'dev_rankings_step6_fused.jsonl',
        OUTPUT_DIR / 'rankings' / slug / 'public_rankings_step6_fused.jsonl',
    ]:
        print(path, 'OK' if path.exists() else 'MISSING')
